# Python for (open) Neuroscience  
### Lecture 2.3 — *Pandas foundations*
#### Module 02 - Scientific Stack
Sara Assecondi

Department of Psychology and Cognitive Sciences

>**Goal of this lecture:** learn the core pandas workflow for loading tabular data, inspecting its structure, and selecting, filtering, and transforming rows and columns confidently.

[Pandas documentation](https://pandas.pydata.org/docs/user_guide/index.html#user-guide)

## Learning goals

By the end of this notebook you should be able to:

- read a `.csv` file into a dataframe
- inspect columns, rows, and data types
- select columns and filter rows
- use boolean masks, `.loc`, and `.iloc`
- perform simple column-wise computations
- sort and summarize data


In [38]:
import numpy as np
import pandas as pd


## Why pandas?

`pandas` is the standard Python library for **tabular data**.

It is especially useful when data have:

- rows = observations / events / trials / subjects
- columns = variables / measurements / labels
- mixed data types (numbers, strings, booleans, timestamps, categories)


## The basic data structures

The two core pandas data structures are:

- `pd.DataFrame`: a 2D table with rows and columns
- `pd.Series`: a 1D labeled array, typically one column of a dataframe


## `pd.Series`
- it is a **one-dimensional labeled array** capable of holding any data type (integers, strings, floating point numbers, Python objects, etc.) 
- the axis labels are collectively referred to as the index
- supports **vectorized operations**


In [39]:
s = pd.Series(np.random.randn(5), index=["a", "b", "c", "d", "e"])

print("s = ",s)
s.index

s =  a    0.434161
b    1.042579
c   -1.353211
d    0.235673
e   -1.364073
dtype: float64


Index(['a', 'b', 'c', 'd', 'e'], dtype='str')

## `pd.DataFrame`

- `DataFrame` is a **2-dimensional labeled** data structure with columns of potentially different types.    
- You can think of it like a spreadsheet or SQL table, or a dict of Series objects.
- Like Series, DataFrame accepts many different kinds of input:

    - Dict of 1D ndarrays, lists, dicts, or Series
    - 2-D numpy.ndarray
    - Structured or record ndarray
    - A Series
    - Another DataFrame

- you can optionally pass **index (row labels)** and **columns (column labels)** arguments

> If you pass an index and / or columns, you are **guaranteeing** the index and / or columns of the resulting DataFrame. Thus, a dict of Series plus a specific index will **discard** all data not matching up to the passed index.

If axis labels are not passed, they will be constructed from the input data based on common sense rules.

In [40]:
df = pd.DataFrame({
    "student_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "name": ["Anna", "Luca", "Marta", "Paolo", "Sara", "Giulia", "Marco", "Elena"],
    "age": [22, 24, 21, 23, 25, 22, 24, 23],
    "city": ["Rome", "Milan", "Rome", "Turin", "Milan", "Naples", "Rome", "Turin"],
    "score_math": [28, 24, 30, 18, 27, 25, np.nan, 29],
    "score_python": [30, 26, 29, 20, 28, np.nan, 24, 30],
    "hours_studied": [12.5, 9.0, 15.0, 4.5, 11.0, 8.0, 6.5, 13.0],
    "passed": [True, True, True, False, True, True, False, True]
})

df

,student_id,name,age,city,score_math,score_python,hours_studied,passed
0,101,Anna,22,Rome,28.0,30.0,12.5,True
1,102,Luca,24,Milan,24.0,26.0,9.0,True
2,103,Marta,21,Rome,30.0,29.0,15.0,True
3,104,Paolo,23,Turin,18.0,20.0,4.5,False
4,105,Sara,25,Milan,27.0,28.0,11.0,True
5,106,Giulia,22,Naples,25.0,NaN,8.0,True
6,107,Marco,24,Rome,NaN,24.0,6.5,False
7,108,Elena,23,Turin,29.0,30.0,13.0,True


### Inspecting a dataframe

A first pass on any new dataframe usually includes:

- `.head()` / `.tail()`
- `.columns`
- `.index`
- `.shape`
- `.dtypes`
- `.info()`
- `.describe()`


In [41]:
df.head()


,student_id,name,age,city,score_math,score_python,hours_studied,passed
0,101,Anna,22,Rome,28.0,30.0,12.5,True
1,102,Luca,24,Milan,24.0,26.0,9.0,True
2,103,Marta,21,Rome,30.0,29.0,15.0,True
3,104,Paolo,23,Turin,18.0,20.0,4.5,False
4,105,Sara,25,Milan,27.0,28.0,11.0,True


In [42]:
df.columns


Index(['student_id', 'name', 'age', 'city', 'score_math', 'score_python',
       'hours_studied', 'passed'],
      dtype='str')

In [ ]:
df.index


In [ ]:
df.shape


In [ ]:
df.dtypes


In [ ]:
df.info()


In [43]:
df.describe(include="all")


,student_id,name,age,city,score_math,score_python,hours_studied,passed
count,8.00000,8,8.000000,8,7.000000,7.000000,8.000000,8
unique,NaN,8,NaN,4,NaN,NaN,NaN,2
top,NaN,Anna,NaN,Rome,NaN,NaN,NaN,True
freq,NaN,1,NaN,3,NaN,NaN,NaN,6
mean,104.50000,NaN,23.000000,NaN,25.857143,26.714286,9.937500,NaN
std,2.44949,NaN,1.309307,NaN,4.059087,3.683942,3.560071,NaN
min,101.00000,NaN,21.000000,NaN,18.000000,20.000000,4.500000,NaN
25%,102.75000,NaN,22.000000,NaN,24.500000,25.000000,7.625000,NaN
50%,104.50000,NaN,23.000000,NaN,27.000000,28.000000,10.000000,NaN
75%,106.25000,NaN,24.000000,NaN,28.500000,29.500000,12.625000,NaN


### Selecting columns

Selecting one column returns a `Series`.  
Selecting multiple columns returns a new `DataFrame`.


In [ ]:
df["age"]


In [ ]:
df[["name", "city"]]


In [ ]:
type(df["student_id"])


### Row selection and boolean masks

When we build a boolean series, we can use it to keep only the rows that match a condition.


In [44]:
boolean_selection = df["age"] == 22
boolean_selection


0     True
1    False
2    False
3    False
4    False
5     True
6    False
7    False
Name: age, dtype: bool

In [45]:
df[boolean_selection]


,student_id,name,age,city,score_math,score_python,hours_studied,passed
0,101,Anna,22,Rome,28.0,30.0,12.5,True
5,106,Giulia,22,Naples,25.0,NaN,8.0,True


Multiple conditions are combined with the same operators used for NumPy arrays:

- `&` for element-wise **and**
- `|` for element-wise **or**
- `~` for element-wise **not**

Remember to wrap each condition in parentheses.


In [46]:
selection_series = (df["city"] == "Rome") & (df["age"] > 21)
df[selection_series]


,student_id,name,age,city,score_math,score_python,hours_studied,passed
0,101,Anna,22,Rome,28.0,30.0,12.5,True
6,107,Marco,24,Rome,NaN,24.0,6.5,False


### Label-based indexing with `.loc`

- primarily **label-based**, but may also be used with a **boolean array**
- `.loc[row_selection, column_selection]` is usually the clearest and most readable way to subset data.


In [47]:
df1 = pd.DataFrame(
    {"age": [20, 25, 30], "city": ["Rome", "Milan", "Turin"]},
    index=["a", "b", "c"]
)
df1

,age,city
a,20,Rome
b,25,Milan
c,30,Turin


In [48]:


# df.loc[row_labels, column_labels]
print(df1.loc["b","age"])

25


In [50]:

boolean_selection = df1["age"]>20
df1.loc[boolean_selection, ["city", "age"]]


,city,age
b,Milan,25
c,Turin,30


### Position-based indexing with `.iloc`
- **integer-location** based indexing for selection by position.
- Use `.iloc` when you want NumPy-like indexing by integer position.


In [ ]:
df.iloc[:5, :2]


### Non-numerical row indexes

Sometimes row labels are meaningful identifiers (trial IDs, subject IDs, filenames, timestamps...).


In [ ]:
non_num_idx_df = df.copy()
non_num_idx_df.index = [f"student_{i}" for i in df.index]
non_num_idx_df.head()


In [ ]:
non_num_idx_df.loc["student_0":"student_2", ["age", "passed"]]


### Basic operations on columns

A few operations come up constantly when working with real datasets.


In [ ]:
df["age"].mean()


In [ ]:
df["passed"].value_counts()


In [ ]:
df.sort_values(by="score_math").head()


In [ ]:
df.assign(mean_score = (df["score_math"] + df["score_python"])/ 2).head()


### Basic filtering

Some methods are nicer than writing long boolean expressions by hand.

- `isin`: “Is this value in this list?”, returns `True/False` for each row.
- `between`: "Is this value between a and b?"

In [ ]:
df["city"].isin(["Rome", "Milan"])


In [ ]:
# When values is a dict, we can pass values to check for each column separately:
df.isin({"city": ["Rome", "Milan"],
        "passed":[True]})

In [ ]:
df["score_math"].between(24, 30)


In [ ]:
df[df["score_math"].between(24, 30)]